In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
from pathlib import Path
import random
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

import torchvision.transforms as transforms
import cv2
import numpy as np
from tqdm import tqdm
from PIL import Image

# CONFIG
IMG_SIZE = 128
BATCH_SIZE = 16
EPOCHS = 5
LR = 1e-3
NUM_WORKERS = 0
SEED = 42

BASE_DRIVE = Path("/content/drive/MyDrive/infosys_internship")
DATASET_DIR = BASE_DRIVE / "dataset"   # your dataset directory (as you said)
ASSIGNMENT_DIR = BASE_DRIVE / "ASSIGNMENT-04"
MODEL_SAVE_PATH = ASSIGNMENT_DIR / "models" / "cnn_segmentation.pth"
OUTPUT_PATH = ASSIGNMENT_DIR / "outputs" / "result.png"
APP_DIR = ASSIGNMENT_DIR / "app"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# create required folders (if not exist)
(ASSIGNMENT_DIR / "models").mkdir(parents=True, exist_ok=True)
(ASSIGNMENT_DIR / "outputs").mkdir(parents=True, exist_ok=True)
APP_DIR.mkdir(parents=True, exist_ok=True)

# reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print(f"DATASET_DIR = {DATASET_DIR}")
print(f"ASSIGNMENT_DIR = {ASSIGNMENT_DIR}")
print(f"DEVICE = {DEVICE}")


DATASET_DIR = /content/drive/MyDrive/infosys_internship/dataset
ASSIGNMENT_DIR = /content/drive/MyDrive/infosys_internship/ASSIGNMENT-04
DEVICE = cpu


In [ ]:
#  CELL 3
from pathlib import Path
from torch.utils.data import DataLoader, Dataset
import torchvision.transforms as transforms
import cv2
import numpy as np

# Config tweak
SKIP_MISSING_MASKS = True   # set False to force error when pairs missing
IMG_EXTS = [".jpg", ".jpeg", ".png", ".bmp"]
MASK_EXTS = [".png", ".jpg", ".jpeg"]

class SimpleDataset(Dataset):
    def __init__(self, dataset_dir, skip_missing=SKIP_MISSING_MASKS):
        self.dataset_dir = Path(dataset_dir)
        if not self.dataset_dir.exists():
            raise FileNotFoundError(f"Dataset directory not found: {self.dataset_dir}")

        # find images recursively
        imgs = []
        for ext in IMG_EXTS:
            imgs.extend(list(self.dataset_dir.rglob(f"*{ext}")))
        imgs = sorted(set(imgs))

        # find masks recursively (any file with mask-like name or mask extension)
        masks = []
        for ext in MASK_EXTS:
            masks.extend(list(self.dataset_dir.rglob(f"*{ext}")))
        masks = sorted(set(masks))

        print(f"Found {len(imgs)} image files and {len(masks)} mask files (searching recursively).")
        if len(imgs) == 0:
            raise FileNotFoundError(f"No image files found under {self.dataset_dir}. Check path or file extensions {IMG_EXTS}.")

        # Build a quick lookup of mask basenames -> list(paths)
        mask_name_map = {}
        mask_filenames = []
        for m in masks:
            name = m.name
            stem = m.stem
            mask_filenames.append(name)
            mask_name_map.setdefault(name, []).append(m)
            mask_name_map.setdefault(stem, []).append(m)

        # heuristic pairing
        pairs = []
        for img_path in imgs:
            stem = img_path.stem
            possible_names = [
                stem,
                f"{stem}_mask",
                f"{stem}-mask",
                f"mask_{stem}",
                f"{stem}_seg",
                f"{stem}_segmentation",
                f"{stem}_gt",
                f"{stem}_annotation",
            ]
            mask_found = None

            # 1) exact filename or stem match
            for pn in possible_names:
                for ext in MASK_EXTS:
                    candidate_name = pn + ext
                    if candidate_name in mask_name_map:
                        mask_found = mask_name_map[candidate_name][0]
                        break
                if mask_found:
                    break

            # 2) direct stem key in map
            if (mask_found is None) and (stem in mask_name_map):
                mask_found = mask_name_map[stem][0]

            # 3) fallback: any mask filename that contains image stem
            if mask_found is None:
                for m in masks:
                    if stem in m.name:
                        mask_found = m
                        break

            # 4) last resort: look in a sibling 'masks' directory with same basename
            if mask_found is None:
                sibling_mask = img_path.parent / "masks" / (stem + ".png")
                if sibling_mask.exists():
                    mask_found = sibling_mask

            if mask_found is not None:
                pairs.append((img_path, mask_found))
            else:
                if not skip_missing:
                    # show diagnostic then raise
                    sample_imgs = [p.name for p in imgs[:10]]
                    sample_masks = [m.name for m in masks[:10]]
                    err_msg = (
                        f"No mask found for image: {img_path.name}\n\n"
                        f"Quick diagnostic:\n"
                        f" - Images (sample): {sample_imgs}\n"
                        f" - Masks  (sample): {sample_masks}\n\n"
                        "Expected mask naming examples: <image>_mask.png, <image>.png (same stem), mask_<image>.png\n"
                        "If masks are in a separate folder, place them under dataset/masks or include them under dataset/.\n"
                        "You can set SKIP_MISSING_MASKS=True to skip images without masks (not recommended if you need supervision)."
                    )
                    raise FileNotFoundError(err_msg)
                # else skip the image silently (useful during debugging)
                # print(f"Skipping (no mask): {img_path.name}")  # uncomment if you want logs

        if len(pairs) == 0:
            # nothing paired -> very helpful diagnostic output
            sample_imgs = [p.name for p in imgs[:20]]
            sample_masks = [m.name for m in masks[:20]]
            msg = (
                "No image-mask pairs were matched by the heuristics.\n\n"
                f"Dataset: {self.dataset_dir}\n"
                f"Total images found: {len(imgs)}\n"
                f"Total masks found: {len(masks)}\n\n"
                f"Sample image files: {sample_imgs}\n"
                f"Sample mask files: {sample_masks}\n\n"
                "Common fixes:\n"
                "- Ensure mask filenames follow a matching convention (e.g. image.jpg <-> image_mask.png OR image.png)\n"
                "- Put masks into a 'masks' folder inside the dataset (dataset/masks)\n"
                "- Rename masks so their stems match the image stems\n"
                "If you want me to keep going despite mismatches, set SKIP_MISSING_MASKS=True and rerun."
            )
            raise FileNotFoundError(msg)

        # finalize
        self.pairs = pairs
        print(f"Dataset ready: {len(self.pairs)} image-mask pairs detected.")
        # show first few pairs
        for a,b in self.pairs[:5]:
            print("  ", a.name, "<-->", b.name)

        # transforms
        self.img_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor()
        ])
        self.mask_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img = cv2.imread(str(img_path))
        if img is None:
            raise RuntimeError(f"Failed to read image {img_path}")
        img = img[..., ::-1]  # BGR->RGB

        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None:
            raise RuntimeError(f"Failed to read mask {mask_path}")
        _, mask = cv2.threshold(mask, 127, 255, cv2.THRESH_BINARY)

        img_t = self.img_transform(img)
        mask_t = self.mask_transform(mask)
        mask_t = (mask_t > 0.5).float()
        return img_t, mask_t

# Create dataset & dataloader (replace previous dataset creation)
dataset = SimpleDataset(DATASET_DIR, skip_missing=SKIP_MISSING_MASKS)
dataset.pairs = dataset.pairs[:2000]
train_loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=="cuda"))
print("DataLoader ready with", len(train_loader), "batches")





Found 11463 image files and 11463 mask files (searching recursively).
Dataset ready: 11463 image-mask pairs detected.
   000000000311.jpg <--> 000000000311.jpg
   000000000509.jpg <--> 000000000509.jpg
   000000000619.jpg <--> 000000000619.jpg
   000000000979.jpg <--> 000000000979.jpg
   000000001118.jpg <--> 000000001118.jpg
DataLoader ready with 125 batches


In [ ]:
class SimpleSegNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2)  # 128 -> 64
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2),  # 64 -> 128
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 1)
        )

    def forward(self, x):
        x = self.enc(x)
        x = self.dec(x)
        return torch.sigmoid(x)  # output in [0,1]


In [ ]:
model = SimpleSegNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCELoss()  # matches sigmoid output

print("Starting training on device:", DEVICE)
for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    for imgs, masks in pbar:
        imgs = imgs.to(DEVICE, dtype=torch.float)
        masks = masks.to(DEVICE, dtype=torch.float)

        optimizer.zero_grad()
        preds = model(imgs)  # B x 1 x H x W
        loss = criterion(preds, masks)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} completed — Avg Loss: {avg_loss:.4f}")

# save model state dict
MODEL_SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)
torch.save(model.state_dict(), str(MODEL_SAVE_PATH))
print(f" Model saved: {MODEL_SAVE_PATH}")


Starting training on device: cpu


Epoch 1/5: 100%|██████████| 125/125 [06:00<00:00,  2.89s/it, loss=0.1151]


Epoch 1 completed — Avg Loss: 0.2756


Epoch 2/5: 100%|██████████| 125/125 [05:07<00:00,  2.46s/it, loss=0.1354]


Epoch 2 completed — Avg Loss: 0.1406


...

Epoch 3 completed — Avg Loss: 0.1274


Epoch 4/5: 100%|██████████| 125/125 [05:01<00:00,  2.41s/it, loss=0.1347]


Epoch 4 completed — Avg Loss: 0.1191


Epoch 5/5: 100%|██████████| 125/125 [05:11<00:00,  2.49s/it, loss=0.0834]

Epoch 5 completed — Avg Loss: 0.1130
 Model saved: /content/drive/MyDrive/infosys_internship/ASSIGNMENT-04/models/cnn_segmentation.pth


In [ ]:
def remove_bg_and_save(img_path, model_path=MODEL_SAVE_PATH, out_path=OUTPUT_PATH, threshold=0.5):
    # load model architecture + weights
    net = SimpleSegNet().to(DEVICE)
    net.load_state_dict(torch.load(str(model_path), map_location=DEVICE))
    net.eval()

    img = cv2.imread(str(img_path))
    if img is None:
        raise RuntimeError(f"Cannot read test image: {img_path}")

    orig = img.copy()
    h, w = img.shape[:2]

    transform = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor()
    ])

    # OpenCV BGR → RGB
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Convert NumPy → PIL
    img_pil = Image.fromarray(img_rgb)

    # Apply transforms
    img_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        pred = net(img_tensor)[0, 0].cpu().numpy()

    mask = (cv2.resize(pred, (w, h)) > threshold).astype(np.uint8)
    result = orig * mask[:, :, None]

    out_path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_path), result)

    print(f"Result saved to: {out_path}")
    return out_path


In [ ]:
app_py = r'''
import streamlit as st
import torch
import cv2
import numpy as np
from torchvision import transforms
import torch.nn as nn
from PIL import Image

IMG_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_PATH = "../models/cnn_segmentation.pth"  # relative to app folder

class SimpleSegNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
        self.dec = nn.Sequential(
            nn.ConvTranspose2d(64, 32, 2, stride=2),
            nn.Conv2d(32, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 1, 1)
        )

    def forward(self, x):
        return torch.sigmoid(self.dec(self.enc(x)))

@st.cache_resource
def load_model():
    model = SimpleSegNet().to(DEVICE)
    model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
    model.eval()
    return model

model = load_model()

st.title("🖼️ Simple Background Removal")
uploaded = st.file_uploader("Upload image (jpg/png)", type=["jpg","png","jpeg"])
if uploaded is not None:
    file_bytes = np.asarray(bytearray(uploaded.read()), dtype=np.uint8)
    img = cv2.imdecode(file_bytes, cv2.IMREAD_COLOR)
    if img is None:
        st.error("Cannot read uploaded image.")
    else:
        h, w = img.shape[:2]
        st.image(img[..., ::-1], channels="RGB", caption="Original")

        transform = transforms.Compose([
            transforms.Resize((IMG_SIZE, IMG_SIZE)),
            transforms.ToTensor()
        ])
        # OpenCV BGR → RGB
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        # Convert to PIL Image
        img_pil = Image.fromarray(img_rgb)

        # Apply transforms
        img_tensor = transform(img_pil).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            mask = model(img_tensor)[0,0].cpu().numpy()
        mask = (cv2.resize(mask, (w, h)) > 0.5).astype(np.uint8)
        result = img * mask[..., None]

        st.subheader("Result")
        st.image(result[..., ::-1], channels="RGB")
        # Provide download
        _, buffer = cv2.imencode('.png', result)
        st.download_button("Download result", buffer.tobytes(), file_name="result.png", mime="image/png")
'''

requirements_txt = """
streamlit
torch
torchvision
opencv-python
numpy
Pillow
"""

# write files to drive
app_file = APP_DIR / "app.py"
req_file = APP_DIR / "requirements.txt"

with open(app_file, "w") as f:
    f.write(app_py.strip())

with open(req_file, "w") as f:
    f.write(requirements_txt.strip())

print(f" Streamlit app files written to: {APP_DIR}")
print("To run locally (or on a VM), `cd ASSIGNMENT-04/app` then `streamlit run app.py`")


 Streamlit app files written to: /content/drive/MyDrive/infosys_internship/ASSIGNMENT-04/app
To run locally (or on a VM), `cd ASSIGNMENT-04/app` then `streamlit run app.py`
